# IFEST 2026 DAC — V1.1.1 AMP-Fixed Strong NLI Baseline

**Task:** headline–article factual consistency (`1 = sesuai`, `0 = tidak sesuai`), evaluated by **Macro F1**.

V1.1 keeps the dataset-driven architecture from V1, but fixes the two problems exposed by the first Kaggle run: an overly restricted threshold search and an unnecessarily slow FP32 / double-forward training loop.

Core design:

1. **mDeBERTa-v3 multilingual NLI** as the semantic cross-encoder, with `content evidence = premise` and `title = hypothesis`.
2. **IDF-weighted evidence retrieval** so long articles are not handled by naive first-512 truncation alone.
3. **Connected-component Stratified Group CV** to keep rows sharing the same normalized title/content in the same fold.
4. **Moderately weighted binary entailment loss** (`w(label=0)=3`) instead of blindly using the full ~9× imbalance ratio.
5. **Same-content pairwise ranking loss**, now computed in the same forward pass as the primary examples.
6. **Exact distribution-aware threshold optimization**, with no fixed `[0.03, 0.97]` search range.
7. **Small factual meta-calibrator** using number, negation, high-IDF term, and lexical coverage features.
8. **Duplicate/graph inference layer** whose reliability is measured on train before it is allowed to override test predictions.
9. **FP16 automatic mixed precision on CUDA**, dynamic padding to multiples of 8, and no gradient checkpointing by default for substantially better T4 throughput.
10. **Per-epoch diagnostics**: class-0/class-1 F1, confusion matrix, probability quantiles, positive/negative probability means, and throughput.

Default configuration is intentionally **3 folds × 2 epochs** for the first reliable Kaggle baseline on **T4 ×2**. Do not increase to 5 folds until this V1.1 produces healthy OOF behavior.

Primary model: `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`.


**V1.1.1 reliability fix:** force FP32 master weights at model load time because the upstream checkpoint config declares FP16 weights; AMP is used only for autocast compute. Includes fail-fast dtype validation before the optimizer is created.

In [1]:
# Environment bootstrap. Kaggle normally already has these packages.
import os, sys, subprocess, importlib.util

SMOKE_TEST = os.environ.get("IFEST_SMOKE_TEST", "0") == "1"
required = ["transformers", "sentencepiece", "sklearn", "pandas", "torch"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing and not SMOKE_TEST:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5", "sentencepiece", "accelerate", "scikit-learn", "pandas"
    ])
    print("Installed missing packages:", missing)
else:
    print("Environment ready. Missing packages:", missing)

Environment ready. Missing packages: []


In [2]:
import os, re, gc, json, math, random, hashlib, unicodedata, warnings, time
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import torch
try:
    import transformers
    TRANSFORMERS_VERSION = transformers.__version__
except ImportError:
    TRANSFORMERS_VERSION = "not-installed (smoke-only)"
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
pd.set_option("display.max_colwidth", 180)

# ---------------- CONFIG ----------------
SEED = 2026
MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
LOCAL_MODEL_PATH = None  # Set to a Kaggle input directory if Internet is disabled.

# First trustworthy baseline budget. Keep this at 3 until OOF behavior is healthy.
N_FOLDS = 3
EPOCHS = 2
MAX_LENGTH = 384

# T4x2-oriented throughput settings.
# DataParallel splits the GLOBAL batch across both GPUs.
BATCH_SIZE = 12          # ~6 primary examples/GPU before capped pairwise additions
EVAL_BATCH_SIZE = 32     # inference has no optimizer/gradient activation cost
GRAD_ACCUM = 2           # effective primary batch ~= 24
MAX_PAIRWISE_PER_BATCH = 4
USE_AMP = True           # CUDA FP16 autocast + GradScaler; model weights stay FP32
USE_GRAD_CHECKPOINTING = False

LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
NEG_CLASS_WEIGHT = 3.0   # sqrt(12960/1437) ~= 3.0
PAIRWISE_LAMBDA = 0.25
PAIRWISE_MARGIN = 0.50
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2

WINDOW_WORDS = 90
WINDOW_STRIDE = 45
TOP_EVIDENCE_WINDOWS = 3
HIGH_IDF_QUANTILE = 0.80

# Graph rules are only enabled if train reliability clears these floors.
CONTENT_RULE_MIN_PRECISION = 0.97
TITLE_RULE_MIN_PRECISION = 0.94
CONTENT_RULE_PROB = 0.02
TITLE_RULE_PROB = 0.05

RUN_TRAINING = not SMOKE_TEST
SAVE_FOLD_MODELS = False  # avoids ~GBs of redundant Kaggle I/O; set True only if checkpoints are needed
OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v1_1_1_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    N_FOLDS = 2
    EPOCHS = 1
    NUM_WORKERS = 0
    USE_AMP = False
    _real_tqdm = tqdm
    def tqdm(iterable=None, *args, **kwargs):
        return iterable


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print(f"PyTorch={torch.__version__} | Transformers={TRANSFORMERS_VERSION}")
print(f"Device={DEVICE} | GPUs={GPU_COUNT} | training={RUN_TRAINING} | AMP={AMP_ENABLED}")
if GPU_COUNT:
    for i in range(GPU_COUNT):
        props = torch.cuda.get_device_properties(i)
        print(i, torch.cuda.get_device_name(i), f"VRAM={props.total_memory/1024**3:.1f} GB")


PyTorch=2.10.0+cu128 | Transformers=5.0.0
Device=cuda | GPUs=2 | training=True | AMP=True
0 Tesla T4 VRAM=14.6 GB
1 Tesla T4 VRAM=14.6 GB


In [3]:
def discover_data_files():
    """Find the competition CSVs on Kaggle or in the supplied local workspace."""
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    if Path("/mnt/data/ifest_dac").exists():
        roots.append(Path("/mnt/data/ifest_dac"))

    train_candidates, test_candidates, sample_candidates = [], [], []
    for root in roots:
        train_candidates.extend(root.rglob("train.csv"))
        test_candidates.extend(root.rglob("test.csv"))
        sample_candidates.extend(root.rglob("sample_submission.csv"))

    def valid(path, required_cols):
        try:
            cols = set(pd.read_csv(path, nrows=2).columns)
            return set(required_cols).issubset(cols)
        except Exception:
            return False

    trains = [p for p in train_candidates if valid(p, ["id", "title", "content", "label"])]
    tests = [p for p in test_candidates if valid(p, ["id", "title", "content"])]
    samples = [p for p in sample_candidates if valid(p, ["id", "label"])]

    if not trains or not tests:
        raise FileNotFoundError(
            "Could not locate IFEST train.csv/test.csv. Add the competition dataset as a Kaggle Input."
        )

    # Prefer train/test from the same directory.
    train_path = trains[0]
    same_dir_test = [p for p in tests if p.parent == train_path.parent]
    test_path = same_dir_test[0] if same_dir_test else tests[0]
    same_dir_sample = [p for p in samples if p.parent == train_path.parent]
    sample_path = same_dir_sample[0] if same_dir_sample else (samples[0] if samples else None)
    return train_path, test_path, sample_path

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = discover_data_files()
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH) if SAMPLE_PATH else pd.DataFrame({"id": test.id, "label": 1})

for df in (train, test):
    df["title"] = df["title"].fillna("").astype(str)
    df["content"] = df["content"].fillna("").astype(str)

if SMOKE_TEST:
    train = train.head(2500).copy().reset_index(drop=True)
    test = test.head(600).copy().reset_index(drop=True)
    sample = sample.head(len(test)).copy().reset_index(drop=True)

print("train:", train.shape, TRAIN_PATH)
print("test :", test.shape, TEST_PATH)
print("labels:")
print(train.label.value_counts().sort_index())
print("minority ratio:", round((train.label == 0).mean(), 4))

assert list(train.columns[:4]) == ["id", "title", "content", "label"]
assert set(train.label.unique()).issubset({0, 1})
assert train.id.is_unique and test.id.is_unique
assert len(sample) == len(test)

train: (14397, 4) /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv
test : (3603, 3) /kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv
labels:
label
0     1437
1    12960
Name: count, dtype: int64
minority ratio: 0.0998


## Text normalization and evidence retrieval

Normalization is intentionally conservative. We keep numbers and negations because they are often exactly what separates `sesuai` from `tidak sesuai`. Only whitespace/Unicode noise and the observed trailing `kumparan com` source artifact are normalized for the model input.

Evidence retrieval uses IDF-weighted title-token overlap. It always keeps the article opening and adds the most title-relevant windows, giving high-IDF entities/numbers more influence than generic COVID vocabulary.

In [4]:
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)*\b")
NEGATIONS = {
    "tidak", "tak", "bukan", "belum", "tanpa", "jangan", "gagal", "mustahil",
    "dilarang", "menolak", "negatif", "tiada", "kurang"
}
TEMPORAL_WORDS = {
    "senin","selasa","rabu","kamis","jumat","sabtu","minggu",
    "januari","februari","maret","april","mei","juni","juli","agustus",
    "september","oktober","november","desember","kemarin","hari","pekan","bulan","tahun"
}
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)


def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_key(text):
    text = clean_ws(text).lower()
    return " ".join(TOKEN_RE.findall(text))


def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()


def tokens(text):
    return TOKEN_RE.findall(normalize_key(text))

train["title_model"] = train.title.map(clean_title_for_model)
test["title_model"] = test.title.map(clean_title_for_model)
train["title_key"] = train.title_model.map(normalize_key)
test["title_key"] = test.title_model.map(normalize_key)
train["content_key"] = train.content.map(normalize_key)
test["content_key"] = test.content.map(normalize_key)
train["pair_key"] = train.title_key + "|||" + train.content_key
test["pair_key"] = test.title_key + "|||" + test.content_key

# IDF from training titles + contents. No labels are used.
doc_freq = Counter()
N_DOCS = len(train)
for t, c in tqdm(zip(train.title_model, train.content), total=N_DOCS, desc="IDF corpus"):
    doc_freq.update(set(tokens(t)) | set(tokens(c)))

idf = {tok: math.log((N_DOCS + 1) / (df + 1)) + 1.0 for tok, df in doc_freq.items()}
IDF_FLOOR = math.log((N_DOCS + 1) / 2) + 1.0
idf_values = np.array(list(idf.values()), dtype=float)
HIGH_IDF_THRESHOLD = float(np.quantile(idf_values, HIGH_IDF_QUANTILE))


def tok_idf(tok):
    return idf.get(tok, IDF_FLOOR)


def window_score(title_toks, window_toks):
    title_unique = set(title_toks)
    if not title_unique:
        return 0.0
    window_set = set(window_toks)
    denom = sum(tok_idf(t) for t in title_unique) + 1e-9
    numer = sum(tok_idf(t) for t in title_unique if t in window_set)
    return numer / denom


def retrieve_evidence(title, content, window_words=WINDOW_WORDS, stride=WINDOW_STRIDE, top_k=TOP_EVIDENCE_WINDOWS):
    title_toks = tokens(title)
    words = clean_ws(content).split()
    if not words:
        return ""
    if len(words) <= window_words:
        return " ".join(words)

    starts = list(range(0, max(1, len(words) - window_words + 1), stride))
    last_start = max(0, len(words) - window_words)
    if starts[-1] != last_start:
        starts.append(last_start)

    windows = [(s, words[s:s + window_words]) for s in starts]
    scores = [(window_score(title_toks, tokens(" ".join(w))), i) for i, (_, w) in enumerate(windows)]

    selected = [0]  # article lead is structurally useful in news
    for _, i in sorted(scores, reverse=True):
        if i not in selected:
            selected.append(i)
        if len(selected) >= top_k:
            break
    selected = sorted(selected)
    return " \n ".join(" ".join(windows[i][1]) for i in selected)


def add_evidence(df):
    out = df.copy()
    out["evidence"] = [
        retrieve_evidence(t, c) for t, c in tqdm(
            zip(out.title_model, out.content), total=len(out), desc="Evidence"
        )
    ]
    return out

train = add_evidence(train)
test = add_evidence(test)

print(train[["title_model", "evidence"]].head(2))

IDF corpus:   0%|          | 0/14397 [00:00<?, ?it/s]

Evidence:   0%|          | 0/14397 [00:00<?, ?it/s]

Evidence:   0%|          | 0/3603 [00:00<?, ?it/s]

                                                                        title_model  \
0                      Tak Lagi Jadi Jubir Yurianto Tak Umumkan Update Harian Covid   
1  Aturan Swab PCR Bagi Penumpang Pesawat ke Kalbar Diperpanjang hingga 28 Februari   

                                                                                                                                                                              evidence  
0  Ada perubahan besar besaran dalam pemaparan data harian perkembangan virus corona Covid 19 di Indonesia Selasa 21 7 Selain tak lagi diumumkan Dirjen P2P Kemenkes Achmad Yuriant...  
1  Aturan Swab PCR Bagi Penumpang Pesawat ke Kalbar Diperpanjang hingga 28 FebruariHi Pontianak Pemerintah Provinsi Kalimantan Barat memperpanjang kebijakan persyaratan kepada pel...  


In [5]:
def lexical_factual_features(df):
    rows = []
    for title, content, evidence in tqdm(
        zip(df.title_model, df.content, df.evidence), total=len(df), desc="Factual features"
    ):
        tt = tokens(title)
        ct = tokens(content)
        et = tokens(evidence)
        tset, cset, eset = set(tt), set(ct), set(et)

        uniq = list(tset)
        denom = max(1, len(uniq))
        title_cov = sum(t in cset for t in uniq) / denom
        evidence_cov = sum(t in eset for t in uniq) / denom
        idf_denom = sum(tok_idf(t) for t in uniq) + 1e-9
        idf_cov = sum(tok_idf(t) for t in uniq if t in cset) / idf_denom
        evidence_idf_cov = sum(tok_idf(t) for t in uniq if t in eset) / idf_denom

        nums = NUM_RE.findall(normalize_key(title))
        ctext = normalize_key(content)
        num_missing = sum(n not in ctext for n in nums) / max(1, len(nums))

        negs = [t for t in tt if t in NEGATIONS]
        neg_missing = sum(n not in cset for n in negs) / max(1, len(negs))

        temps = [t for t in tt if t in TEMPORAL_WORDS]
        temporal_missing = sum(t not in cset for t in temps) / max(1, len(temps))

        hi = [t for t in uniq if tok_idf(t) >= HIGH_IDF_THRESHOLD and not t.isdigit()]
        high_idf_missing = sum(t not in cset for t in hi) / max(1, len(hi))

        union = len(tset | cset)
        jaccard = len(tset & cset) / max(1, union)
        rows.append({
            "title_cov": title_cov,
            "evidence_cov": evidence_cov,
            "idf_cov": idf_cov,
            "evidence_idf_cov": evidence_idf_cov,
            "retrieval_gain": evidence_idf_cov - window_score(tt, ct[:WINDOW_WORDS]),
            "num_count": len(nums),
            "num_missing": num_missing,
            "neg_count": len(negs),
            "neg_missing": neg_missing,
            "temporal_count": len(temps),
            "temporal_missing": temporal_missing,
            "high_idf_missing": high_idf_missing,
            "jaccard": jaccard,
            "title_words": len(tt),
            "content_words": len(ct),
            "evidence_words": len(et),
            "length_ratio": len(tt) / max(1, len(ct)),
        })
    return pd.DataFrame(rows, index=df.index)

train_feat = lexical_factual_features(train)
test_feat = lexical_factual_features(test)
META_FEATURES = list(train_feat.columns)
print(train_feat.describe().T[["mean", "std", "min", "max"]])

Factual features:   0%|          | 0/14397 [00:00<?, ?it/s]

Factual features:   0%|          | 0/3603 [00:00<?, ?it/s]

                        mean         std       min          max
title_cov           0.798746    0.136790  0.000000     1.000000
evidence_cov        0.788544    0.139585  0.000000     1.000000
idf_cov             0.759067    0.164487  0.000000     1.000000
evidence_idf_cov    0.750890    0.165643  0.000000     1.000000
retrieval_gain      0.080047    0.126312  0.000000     1.000000
num_count           0.819268    1.149487  0.000000     8.000000
num_missing         0.016913    0.117899  0.000000     1.000000
neg_count           0.118705    0.336504  0.000000     3.000000
neg_missing         0.038179    0.190960  0.000000     1.000000
temporal_count      0.160450    0.380966  0.000000     3.000000
temporal_missing    0.042231    0.200575  0.000000     1.000000
high_idf_missing    0.013637    0.115466  0.000000     1.000000
jaccard             0.051286    0.023968  0.000000     0.363636
title_words         9.948461    1.859166  3.000000    22.000000
content_words     309.898451  169.571327

## Duplicate-aware connected-component groups

Rows are connected when they share an exact normalized title **or** exact normalized content. Every connected component is kept entirely inside one fold. This intentionally removes the strongest duplicate shortcut from semantic validation, so OOF Macro F1 reflects genuine generalization.

In [6]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.r[ra] < self.r[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]:
            self.r[ra] += 1


def build_component_groups(df):
    uf = UnionFind(len(df))
    for col in ["title_key", "content_key"]:
        first = {}
        for i, key in enumerate(df[col].tolist()):
            if key in first:
                uf.union(i, first[key])
            else:
                first[key] = i
    roots = [uf.find(i) for i in range(len(df))]
    root_to_group = {r: j for j, r in enumerate(dict.fromkeys(roots))}
    return np.array([root_to_group[r] for r in roots], dtype=np.int32)

groups = build_component_groups(train)
train["cv_group"] = groups
print("Connected components:", train.cv_group.nunique())
print("Rows in non-singleton components:", int(train.groupby("cv_group").size().loc[lambda s: s > 1].sum()))

cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(cv.split(train, train.label.values, groups=groups))
for fold, (tr_idx, va_idx) in enumerate(splits):
    tr_groups = set(groups[tr_idx]); va_groups = set(groups[va_idx])
    assert tr_groups.isdisjoint(va_groups)
    print(
        f"fold={fold} train={len(tr_idx):5d} valid={len(va_idx):5d} "
        f"valid_neg={(train.iloc[va_idx].label==0).mean():.3f}"
    )

Connected components: 11525
Rows in non-singleton components: 5419
fold=0 train= 9599 valid= 4798 valid_neg=0.099
fold=1 train= 9590 valid= 4807 valid_neg=0.096
fold=2 train= 9605 valid= 4792 valid_neg=0.104


## Empirical graph-rule reliability

The graph layer is **not** learned from strict group-CV OOF scores because strict CV deliberately removes duplicate overlap. Instead, its reliability is estimated with leave-one-out counts on train, then only rules above conservative precision floors are enabled for test-time post-processing.

In [7]:
def estimate_graph_rule_reliability(df):
    pair_label_counts = defaultdict(Counter)
    content_pos_title_counts = defaultdict(Counter)
    title_pos_content_counts = defaultdict(Counter)

    for r in df.itertuples():
        pair_label_counts[r.pair_key][int(r.label)] += 1
        if int(r.label) == 1:
            content_pos_title_counts[r.content_key][r.title_key] += 1
            title_pos_content_counts[r.title_key][r.content_key] += 1

    records = []
    for r in df.itertuples():
        y = int(r.label)
        # 1) Exact pair, excluding this row itself.
        cnt = pair_label_counts[r.pair_key].copy()
        cnt[y] -= 1
        if cnt[y] <= 0:
            cnt.pop(y, None)
        if sum(cnt.values()) > 0 and len(cnt) == 1:
            pred = next(iter(cnt.keys()))
            records.append(("exact_pair", pred, y))
            continue

        # 2) Same content has another positive headline after excluding current row.
        ccounts = content_pos_title_counts[r.content_key].copy()
        if y == 1:
            ccounts[r.title_key] -= 1
            if ccounts[r.title_key] <= 0:
                ccounts.pop(r.title_key, None)
        if sum(ccounts.values()) > 0 and r.title_key not in ccounts:
            records.append(("content_pos_different_title", 0, y))
            continue

        # 3) Same title has another positive content after excluding current row.
        tcounts = title_pos_content_counts[r.title_key].copy()
        if y == 1:
            tcounts[r.content_key] -= 1
            if tcounts[r.content_key] <= 0:
                tcounts.pop(r.content_key, None)
        if sum(tcounts.values()) > 0 and r.content_key not in tcounts:
            records.append(("title_pos_different_content", 0, y))

    res = pd.DataFrame(records, columns=["rule", "pred", "y"])
    stats = []
    for rule, g in res.groupby("rule"):
        stats.append({
            "rule": rule,
            "coverage": len(g),
            "accuracy": (g.pred == g.y).mean(),
            "precision_when_predicting_0": ((g.y == 0).mean() if (g.pred == 0).all() else np.nan),
        })
    return pd.DataFrame(stats).set_index("rule") if stats else pd.DataFrame()

graph_stats = estimate_graph_rule_reliability(train)
display(graph_stats)

content_rule_precision = float(graph_stats.loc["content_pos_different_title", "precision_when_predicting_0"]) \
    if "content_pos_different_title" in graph_stats.index else 0.0
title_rule_precision = float(graph_stats.loc["title_pos_different_content", "precision_when_predicting_0"]) \
    if "title_pos_different_content" in graph_stats.index else 0.0
ENABLE_CONTENT_RULE = content_rule_precision >= CONTENT_RULE_MIN_PRECISION
ENABLE_TITLE_RULE = title_rule_precision >= TITLE_RULE_MIN_PRECISION
print("content rule enabled:", ENABLE_CONTENT_RULE, content_rule_precision)
print("title rule enabled  :", ENABLE_TITLE_RULE, title_rule_precision)

,coverage,accuracy,precision_when_predicting_0
rule,,,
content_pos_different_title,924,0.984848,0.984848
exact_pair,3620,1.000000,NaN
title_pos_different_content,40,0.800000,0.800000


content rule enabled: True 0.9848484848484849
title rule enabled  : False 0.8


## Helper self-tests

These tests protect the parts most likely to silently introduce leakage or destroy factual signals: normalization, evidence retrieval, component grouping, and graph-rule precedence.

In [8]:
# Minimal behavioral tests.
assert normalize_key("  Bali\xa0  4.617 ") == "bali 4 617"
assert "belum" in tokens("Pemerintah belum capai target")
assert "4" in tokens("Kasus 4.617 orang")
assert len(retrieve_evidence("Bali PCR", "Bali PCR berlaku hari ini")) > 0

_toy = pd.DataFrame({
    "title_key": ["a", "b", "c"],
    "content_key": ["x", "x", "z"]
})
_tg = build_component_groups(_toy)
assert _tg[0] == _tg[1] and _tg[2] != _tg[0]

# Train/test exact-pair overlap audit, useful for the final graph layer.
train_pair_labels = train.groupby("pair_key").label.agg(lambda s: tuple(sorted(set(map(int, s)))))
conflicting_pairs = int((train_pair_labels.map(len) > 1).sum())
known_test_pairs = test.pair_key.isin(train_pair_labels.index).sum()
print("conflicting train exact pairs:", conflicting_pairs)
print("test exact pairs seen in train:", int(known_test_pairs), f"({known_test_pairs/len(test):.2%})")
assert conflicting_pairs == 0

conflicting train exact pairs: 0
test exact pairs seen in train: 927 (25.73%)


## NLI model and efficient pairwise training

The pretrained head has three NLI outputs in model-card order: **entailment, neutral, contradiction**. We convert them into a binary match logit:

`match_logit = entailment_logit - logsumexp(neutral_logit, contradiction_logit)`

This treats both neutral/different-event and contradiction as `label=0` while preserving the NLI initialization.

V1.1 changes the implementation without changing the objective:

- primary and opposite-title pairwise examples are concatenated into **one tokenizer batch and one Transformer forward pass**;
- pairwise additions are capped per batch to prevent random OOM spikes;
- CUDA uses **FP16 autocast + GradScaler** while keeping FP32 master model weights;
- padding is rounded to a multiple of 8 for Tensor Core friendly shapes;
- gradient checkpointing is disabled by default because on T4 it trades memory for substantial extra compute;
- threshold selection evaluates the **actual score distribution**, so very small or very large calibrated probabilities are handled correctly.


In [9]:
class NewsPairDataset(Dataset):
    def __init__(self, df, pair_titles=None):
        self.df = df.reset_index(drop=False).rename(columns={"index": "orig_index"})
        self.pair_titles = pair_titles or {}
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {
            "orig_index": int(r.orig_index),
            "evidence": r.evidence,
            "title": r.title_model,
            "label": int(r.label) if "label" in self.df.columns else -1,
            "pair_title": self.pair_titles.get(int(r.orig_index)),
        }


def build_pair_titles(train_fold):
    """Map original row index -> opposite-label headline sharing the exact same content."""
    pos = defaultdict(list); neg = defaultdict(list)
    for idx, r in train_fold.iterrows():
        (pos if int(r.label) == 1 else neg)[r.content_key].append((idx, r.title_model))
    pair_titles = {}
    for key in set(pos) & set(neg):
        positive_title = pos[key][0][1]
        negative_title = neg[key][0][1]
        for idx, _ in pos[key]:
            pair_titles[int(idx)] = negative_title
        for idx, _ in neg[key]:
            pair_titles[int(idx)] = positive_title
    return pair_titles


def make_collator(tokenizer):
    def collate(batch):
        premises = [x["evidence"] for x in batch]
        hypotheses = [x["title"] for x in batch]

        pair_positions = [i for i, x in enumerate(batch) if x["pair_title"] is not None]
        # Batch order is already randomized by the train DataLoader, so this cap is an unbiased
        # practical limiter for the extra pairwise work and prevents occasional doubled batches.
        pair_positions = pair_positions[:MAX_PAIRWISE_PER_BATCH]

        all_premises = premises + [premises[i] for i in pair_positions]
        all_hypotheses = hypotheses + [batch[i]["pair_title"] for i in pair_positions]
        enc = tokenizer(
            all_premises,
            all_hypotheses,
            padding=True,
            truncation="only_first",
            max_length=MAX_LENGTH,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        return {
            "enc": enc,
            "labels": torch.tensor([x["label"] for x in batch], dtype=torch.float32),
            "orig_index": torch.tensor([x["orig_index"] for x in batch], dtype=torch.long),
            "main_count": len(batch),
            "pair_positions": torch.tensor(pair_positions, dtype=torch.long),
        }
    return collate


def resolve_model_source():
    if LOCAL_MODEL_PATH:
        return LOCAL_MODEL_PATH
    # If the HF model was attached as a Kaggle Dataset, prefer the local copy.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cfg in kaggle_input.rglob("config.json"):
            parent = cfg.parent
            name = str(parent).lower()
            if "mdeberta" in name and (parent / "tokenizer_config.json").exists():
                print("Using local Kaggle model:", parent)
                return str(parent)
    return MODEL_NAME


def floating_parameter_dtypes(model):
    return sorted({str(p.dtype) for p in model.parameters() if p.is_floating_point()})


def ensure_fp32_master_weights(model, verbose=True):
    """GradScaler requires trainable master parameters/grads to remain FP32."""
    bad = [p for p in model.parameters() if p.is_floating_point() and p.dtype != torch.float32]
    if bad:
        if verbose:
            print(f"Casting {len(bad)} non-FP32 floating parameter tensors to FP32 master weights")
        model.float()
    remaining = [p.dtype for p in model.parameters() if p.is_floating_point() and p.dtype != torch.float32]
    if remaining:
        raise RuntimeError(f"FP32 master-weight invariant failed; remaining dtypes={sorted(set(map(str, remaining)))}")
    if verbose:
        print("Master parameter dtypes:", floating_parameter_dtypes(model))
    return model


def load_tokenizer_and_model():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    source = resolve_model_source()
    tokenizer = AutoTokenizer.from_pretrained(source, use_fast=True)

    # IMPORTANT: the upstream checkpoint config declares torch_dtype=float16.
    # Modern Transformers may therefore instantiate FP16 weights by default. GradScaler must
    # instead operate on FP32 master weights while autocast supplies FP16 compute on the T4s.
    try:
        model = AutoModelForSequenceClassification.from_pretrained(source, dtype=torch.float32)
    except TypeError:
        # Compatibility with older Transformers versions where the keyword was torch_dtype.
        model = AutoModelForSequenceClassification.from_pretrained(source, torch_dtype=torch.float32)
    model = ensure_fp32_master_weights(model)

    if model.config.num_labels != 3:
        raise ValueError(f"Expected 3-label NLI head, got {model.config.num_labels}")
    print("NLI id2label:", model.config.id2label)
    model.config.use_cache = False
    if USE_GRAD_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing: ON")
    else:
        print("Gradient checkpointing: OFF")
    return tokenizer, model


def match_logits(nli_logits):
    entail = nli_logits[:, 0]
    nonmatch = torch.logsumexp(nli_logits[:, 1:3], dim=-1)
    return entail - nonmatch


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def move_encoding(enc, device):
    return {k: v.to(device, non_blocking=True) for k, v in enc.items()}


def amp_context():
    if AMP_ENABLED:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler():
    # Compatible with both newer and older Kaggle PyTorch builds.
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)



# Regression test for the V1.1 Kaggle crash: FP16 master parameters must be promoted before AMP training.
_dtype_probe = nn.Linear(2, 2).half()
ensure_fp32_master_weights(_dtype_probe, verbose=False)
assert all(p.dtype == torch.float32 for p in _dtype_probe.parameters())
_probe_loss = _dtype_probe(torch.randn(2, 2)).sum()
_probe_loss.backward()
assert all(p.grad is None or p.grad.dtype == torch.float32 for p in _dtype_probe.parameters())
del _probe_loss, _dtype_probe

def best_threshold(y_true, probs):
    """Exact Macro-F1 threshold search over the observed probability distribution."""
    y = np.asarray(y_true, dtype=np.int8)
    p = np.asarray(probs, dtype=np.float64)
    if y.ndim != 1 or p.ndim != 1 or len(y) != len(p):
        raise ValueError("y_true and probs must be aligned 1-D arrays")
    if len(y) == 0 or not np.isfinite(p).all():
        raise ValueError("Empty or non-finite probability array")

    order = np.argsort(-p, kind="mergesort")
    ps = p[order]
    ys = y[order]
    pos_total = float(ys.sum())
    neg_total = float(len(ys) - ys.sum())

    tp_cum = np.cumsum(ys, dtype=np.float64)
    fp_cum = np.cumsum(1 - ys, dtype=np.float64)
    # Evaluate only after the last row of each equal-score block.
    boundaries = np.flatnonzero(np.r_[ps[1:] != ps[:-1], True])

    tp = tp_cum[boundaries]
    fp = fp_cum[boundaries]
    fn = pos_total - tp
    tn = neg_total - fp
    f1_pos = 2.0 * tp / np.maximum(1.0, 2.0 * tp + fp + fn)
    f1_neg = 2.0 * tn / np.maximum(1.0, 2.0 * tn + fp + fn)
    scores = 0.5 * (f1_pos + f1_neg)
    thresholds = ps[boundaries]

    # Also evaluate the all-negative operating point (threshold just above max score).
    all_neg_f1 = 0.5 * (0.0 + (2.0 * neg_total / max(1.0, 2.0 * neg_total + pos_total)))
    if all_neg_f1 > float(scores.max()):
        return float(np.nextafter(ps[0], np.inf)), float(all_neg_f1)

    i = int(np.argmax(scores))
    return float(thresholds[i]), float(scores[i])


def probability_diagnostics(y_true, probs, prefix=""):
    y = np.asarray(y_true, dtype=np.int8)
    p = np.asarray(probs, dtype=np.float64)
    t, macro = best_threshold(y, p)
    pred = (p >= t).astype(np.int8)
    f1s = f1_score(y, pred, labels=[0, 1], average=None, zero_division=0)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    q = np.quantile(p, [0.0, 0.01, 0.10, 0.50, 0.90, 0.99, 1.0])
    pos_mean = float(p[y == 1].mean()) if np.any(y == 1) else float("nan")
    neg_mean = float(p[y == 0].mean()) if np.any(y == 0) else float("nan")
    print(
        f"{prefix} macro_f1={macro:.6f} t={t:.6g} "
        f"F1_0={f1s[0]:.4f} F1_1={f1s[1]:.4f} "
        f"mean_p(y0)={neg_mean:.5f} mean_p(y1)={pos_mean:.5f}"
    )
    print(
        f"{prefix} p[min,p01,p10,p50,p90,p99,max]="
        + ", ".join(f"{v:.5f}" for v in q)
    )
    print(f"{prefix} confusion_matrix=[[{cm[0,0]}, {cm[0,1]}], [{cm[1,0]}, {cm[1,1]}]]")
    return {
        "threshold": float(t),
        "macro_f1": float(macro),
        "f1_0": float(f1s[0]),
        "f1_1": float(f1s[1]),
        "mean_prob_y0": neg_mean,
        "mean_prob_y1": pos_mean,
        "prob_min": float(q[0]),
        "prob_p01": float(q[1]),
        "prob_p10": float(q[2]),
        "prob_p50": float(q[3]),
        "prob_p90": float(q[4]),
        "prob_p99": float(q[5]),
        "prob_max": float(q[6]),
    }


def predict_model(model, loader, desc="predict"):
    model.eval()
    probs, indices = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, leave=False, desc=desc):
            enc = move_encoding(batch["enc"], DEVICE)
            main_count = int(batch["main_count"])
            with amp_context():
                logits = model(**enc, return_dict=False)[0][:main_count]
                p = torch.sigmoid(match_logits(logits))
            probs.append(p.float().cpu().numpy())
            indices.append(batch["orig_index"].numpy())
    return np.concatenate(indices), np.concatenate(probs)


def loader_extra_kwargs(shuffle=False):
    kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=2)
    return kw


In [10]:
def train_one_fold(fold, tr_idx, va_idx, test_df):
    from transformers import get_cosine_schedule_with_warmup

    tr_df = train.iloc[tr_idx].copy()
    va_df = train.iloc[va_idx].copy()
    pair_titles = build_pair_titles(tr_df)
    print(f"Fold {fold}: pairwise-supervised rows={len(pair_titles)}")

    tokenizer, base_model = load_tokenizer_and_model()
    collate = make_collator(tokenizer)
    tr_loader = DataLoader(
        NewsPairDataset(tr_df, pair_titles), batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate, **loader_extra_kwargs(shuffle=True)
    )
    va_loader = DataLoader(
        NewsPairDataset(va_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        collate_fn=collate, **loader_extra_kwargs()
    )
    te_loader = DataLoader(
        NewsPairDataset(test_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        collate_fn=collate, **loader_extra_kwargs()
    )

    # Small zero-shot diagnostic only. It catches label/probability-order failures cheaply
    # before spending an epoch, but is NOT used for checkpoint/model selection.
    diag_df = va_df.sample(n=min(512, len(va_df)), random_state=SEED + fold)
    diag_loader = DataLoader(
        NewsPairDataset(diag_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        collate_fn=collate, **loader_extra_kwargs()
    )

    base_model.to(DEVICE)
    # Moving devices must not alter the FP32 master-weight invariant. Fail here, before any optimizer step.
    ensure_fp32_master_weights(base_model)
    model = nn.DataParallel(base_model) if GPU_COUNT > 1 else base_model
    if GPU_COUNT > 1:
        print(f"Using DataParallel on {GPU_COUNT} GPUs")

    z_idx, z_prob = predict_model(model, diag_loader, desc=f"fold {fold} zero-shot diagnostic")
    z_map = dict(zip(z_idx.tolist(), z_prob.tolist()))
    z_aligned = np.array([z_map[int(i)] for i in diag_df.index], dtype=float)
    probability_diagnostics(diag_df.label.values, z_aligned, prefix=f"fold={fold} zero-shot")
    del diag_loader

    master_dtypes = floating_parameter_dtypes(unwrap_model(model))
    if master_dtypes != ["torch.float32"]:
        raise RuntimeError(f"Refusing to create optimizer with non-FP32 master weights: {master_dtypes}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = make_grad_scaler()
    update_steps_per_epoch = math.ceil(len(tr_loader) / GRAD_ACCUM)
    total_steps = update_steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_f1 = -1.0
    best_t = 0.5
    best_state = None
    history = []

    optimizer.zero_grad(set_to_none=True)
    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()
        model.train()
        running = []
        pair_rows_seen = 0
        pbar = tqdm(tr_loader, desc=f"fold {fold} epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            enc = move_encoding(batch["enc"], DEVICE)
            y = batch["labels"].to(DEVICE, non_blocking=True)
            main_count = int(batch["main_count"])
            pair_positions = batch["pair_positions"].to(DEVICE, non_blocking=True)
            pair_rows_seen += int(pair_positions.numel())

            with amp_context():
                all_logits = model(**enc, return_dict=False)[0]
                main_mlogit = match_logits(all_logits[:main_count])

                per_item = F.binary_cross_entropy_with_logits(main_mlogit, y, reduction="none")
                weights = torch.where(
                    y < 0.5,
                    torch.full_like(y, NEG_CLASS_WEIGHT),
                    torch.ones_like(y),
                )
                cls_loss = (per_item * weights).mean()

                rank_loss = torch.zeros((), device=DEVICE)
                if pair_positions.numel() > 0:
                    pair_mlogit = match_logits(all_logits[main_count:])
                    current = main_mlogit[pair_positions]
                    current_y = y[pair_positions]
                    sign = torch.where(current_y > 0.5, torch.ones_like(current_y), -torch.ones_like(current_y))
                    diff = sign * (current - pair_mlogit)
                    rank_loss = F.relu(PAIRWISE_MARGIN - diff).mean()

                loss = cls_loss + PAIRWISE_LAMBDA * rank_loss

            scaler.scale(loss / GRAD_ACCUM).backward()
            running.append(float(loss.detach().float().cpu()))

            is_update = ((step + 1) % GRAD_ACCUM == 0) or (step + 1 == len(tr_loader))
            if is_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                # When GradScaler detects inf/nan it skips optimizer.step and reduces scale.
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if step % 25 == 0:
                pbar.set_postfix(
                    loss=f"{np.mean(running[-50:]):.4f}",
                    scale=f"{scaler.get_scale():.0f}",
                )

        va_indices, va_probs = predict_model(model, va_loader, desc=f"fold {fold} validation")
        val_map = dict(zip(va_indices.tolist(), va_probs.tolist()))
        aligned = np.array([val_map[int(i)] for i in va_df.index], dtype=float)
        diag = probability_diagnostics(va_df.label.values, aligned, prefix=f"fold={fold} epoch={epoch+1}")

        elapsed = time.perf_counter() - epoch_start
        samples_per_sec = len(tr_df) / max(elapsed, 1e-9)
        diag.update({
            "fold": fold,
            "epoch": epoch + 1,
            "train_loss": float(np.mean(running)),
            "epoch_seconds": float(elapsed),
            "primary_samples_per_sec": float(samples_per_sec),
            "pair_rows_used": int(pair_rows_seen),
            "amp_scale": float(scaler.get_scale()),
        })
        history.append(diag)
        print(
            f"fold={fold} epoch={epoch+1} time={elapsed/60:.2f} min "
            f"throughput={samples_per_sec:.1f} primary rows/s pair_rows={pair_rows_seen}"
        )

        if diag["macro_f1"] > best_f1:
            best_f1 = diag["macro_f1"]
            best_t = diag["threshold"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in unwrap_model(model).state_dict().items()
            }

    if best_state is None:
        raise RuntimeError("No best model state was selected")

    unwrap_model(model).load_state_dict(best_state)
    va_indices, va_probs = predict_model(model, va_loader, desc=f"fold {fold} best validation")
    te_indices, te_probs = predict_model(model, te_loader, desc=f"fold {fold} test")

    ckpt_path = OUTPUT_DIR / f"ifest_v1_1_1_fold{fold}.pt"
    if SAVE_FOLD_MODELS:
        torch.save({
            "model_name": MODEL_NAME,
            "state_dict": best_state,
            "fold": fold,
            "best_macro_f1": best_f1,
            "best_threshold": best_t,
            "config": {
                "max_length": MAX_LENGTH,
                "batch_size": BATCH_SIZE,
                "grad_accum": GRAD_ACCUM,
                "use_amp": AMP_ENABLED,
                "neg_class_weight": NEG_CLASS_WEIGHT,
                "pairwise_lambda": PAIRWISE_LAMBDA,
                "pairwise_margin": PAIRWISE_MARGIN,
                "max_pairwise_per_batch": MAX_PAIRWISE_PER_BATCH,
            }
        }, ckpt_path)
        print("saved", ckpt_path)

    del model, base_model, tokenizer, tr_loader, va_loader, te_loader
    del optimizer, scheduler, scaler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return va_indices, va_probs, te_indices, te_probs, history


## Connected-component OOF training

This is the main training cell. Default is **3 folds × 2 epochs** and is intentionally the version to run first on Kaggle T4 ×2.

V1.1 now prints a zero-shot sanity diagnostic before each fold and detailed probability diagnostics after every epoch. A healthy run should no longer be judged from a threshold forced into a fixed numeric range. If the model's scores live near `0.002` or `0.98`, the exact threshold optimizer will still evaluate them correctly.

Only after V1.1 gives stable class-0/class-1 OOF behavior should a 5-fold final run be considered.


In [11]:
oof_semantic = np.full(len(train), np.nan, dtype=np.float32)
test_semantic_folds = []
train_history = []

if RUN_TRAINING:
    for fold, (tr_idx, va_idx) in enumerate(splits):
        va_indices, va_probs, te_indices, te_probs, hist = train_one_fold(fold, tr_idx, va_idx, test)
        oof_semantic[va_indices] = va_probs.astype(np.float32)
        test_map = dict(zip(te_indices.tolist(), te_probs.tolist()))
        test_aligned = np.array([test_map[int(i)] for i in test.index], dtype=np.float32)
        test_semantic_folds.append(test_aligned)
        train_history.extend(hist)

    assert not np.isnan(oof_semantic).any()
    test_semantic = np.mean(np.stack(test_semantic_folds), axis=0)
else:
    # CPU-safe placeholder used only by optional plumbing smoke validation.
    # Normal Kaggle execution has RUN_TRAINING=True and never uses this branch.
    print("SMOKE TEST: skipping transformer training")
    oof_semantic = train_feat["evidence_idf_cov"].to_numpy(dtype=np.float32)
    test_semantic = test_feat["evidence_idf_cov"].to_numpy(dtype=np.float32)

semantic_diag = probability_diagnostics(train.label.values, oof_semantic, prefix="Semantic strict OOF")
base_t = semantic_diag["threshold"]
base_f1 = semantic_diag["macro_f1"]
print(classification_report(train.label.values, (oof_semantic >= base_t).astype(int), digits=4))


Fold 0: pairwise-supervised rows=1333


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

...

fold 2 validation:   0%|          | 0/150 [00:00<?, ?it/s]

fold=2 epoch=2 macro_f1=0.846954 t=0.640588 F1_0=0.7254 F1_1=0.9685 mean_p(y0)=0.31151 mean_p(y1)=0.96796
fold=2 epoch=2 p[min,p01,p10,p50,p90,p99,max]=0.00006, 0.00017, 0.58585, 0.99897, 0.99943, 0.99950, 0.99954
fold=2 epoch=2 confusion_matrix=[[358, 142], [129, 4163]]
fold=2 epoch=2 time=7.82 min throughput=20.5 primary rows/s pair_rows=1290


fold 2 best validation:   0%|          | 0/150 [00:00<?, ?it/s]

fold 2 test:   0%|          | 0/113 [00:00<?, ?it/s]

Semantic strict OOF macro_f1=0.836544 t=0.339001 F1_0=0.7044 F1_1=0.9687 mean_p(y0)=0.30073 mean_p(y1)=0.96179
Semantic strict OOF p[min,p01,p10,p50,p90,p99,max]=0.00004, 0.00015, 0.49536, 0.99875, 0.99938, 0.99949, 0.99957
Semantic strict OOF confusion_matrix=[[971, 466], [349, 12611]]
              precision    recall  f1-score   support

           0     0.7356    0.6757    0.7044      1437
           1     0.9644    0.9731    0.9687     12960

    accuracy                         0.9434     14397
   macro avg     0.8500    0.8244    0.8365     14397
weighted avg     0.9415    0.9434    0.9423     14397



## OOF factual meta-calibration

The meta-model never sees in-sample transformer probabilities. It is fitted only on **OOF semantic scores** plus row-local factual features. The duplicate graph stays separate so strict CV remains interpretable.

In [12]:
X_meta = train_feat[META_FEATURES].copy()
X_meta.insert(0, "semantic_prob", oof_semantic)
X_test_meta = test_feat[META_FEATURES].copy()
X_test_meta.insert(0, "semantic_prob", test_semantic)


def make_meta_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=SEED)
    )

# Cross-fitted meta predictions. This avoids evaluating the logistic layer on rows it fitted.
oof_meta = np.full(len(train), np.nan, dtype=np.float64)
for fold, (mtr, mva) in enumerate(splits):
    fold_meta = make_meta_model()
    fold_meta.fit(X_meta.iloc[mtr], train.label.values[mtr])
    oof_meta[mva] = fold_meta.predict_proba(X_meta.iloc[mva])[:, 1]

assert not np.isnan(oof_meta).any()
meta_diag = probability_diagnostics(train.label.values, oof_meta, prefix="Cross-fitted Meta OOF")
meta_t = meta_diag["threshold"]
meta_f1 = meta_diag["macro_f1"]
print(classification_report(train.label.values, (oof_meta >= meta_t).astype(int), digits=4))

# Final meta model consumes only OOF base-model scores for train, then predicts averaged test scores.
meta = make_meta_model()
meta.fit(X_meta, train.label.values)
test_meta = meta.predict_proba(X_test_meta)[:, 1]


Cross-fitted Meta OOF macro_f1=0.840644 t=0.653375 F1_0=0.7125 F1_1=0.9688 mean_p(y0)=0.44356 mean_p(y1)=0.95048
Cross-fitted Meta OOF p[min,p01,p10,p50,p90,p99,max]=0.03531, 0.11568, 0.71489, 0.97513, 0.98692, 0.99255, 1.00000
Cross-fitted Meta OOF confusion_matrix=[[1005, 432], [379, 12581]]
              precision    recall  f1-score   support

           0     0.7262    0.6994    0.7125      1437
           1     0.9668    0.9708    0.9688     12960

    accuracy                         0.9437     14397
   macro avg     0.8465    0.8351    0.8406     14397
weighted avg     0.9428    0.9437    0.9432     14397



## Final graph inference layer

Rule order:

1. exact train `title+content` pair → deterministic known label (only because train has no contradictory exact-pair labels),
2. same content with a different known-positive title → strong `label=0` clamp if measured precision is high enough,
3. same title with a different known-positive content → weaker `label=0` clamp if measured precision is high enough.

The semantic/meta probability remains untouched for rows not covered by these rules.

In [13]:
def build_graph_reference(ref_df):
    pair_to_label = {}
    for key, g in ref_df.groupby("pair_key"):
        labs = sorted(set(map(int, g.label)))
        if len(labs) == 1:
            pair_to_label[key] = labs[0]

    content_to_pos_titles = defaultdict(set)
    title_to_pos_contents = defaultdict(set)
    for r in ref_df.loc[ref_df.label == 1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label, content_to_pos_titles, title_to_pos_contents


def apply_graph_layer(query_df, probs, ref_df):
    """
    Return display probabilities plus explicit forced labels.

    Forced labels are intentionally separate from probability clamping: V1.1's exact threshold
    can legitimately be below 0.02, so a fixed probability clamp must never be relied on to
    enforce a graph decision.
    """
    probs = np.asarray(probs, dtype=float).copy()
    pair_to_label, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    rule_used = np.array(["semantic"] * len(query_df), dtype=object)
    forced_label = np.full(len(query_df), -1, dtype=np.int8)

    for pos, r in enumerate(query_df.itertuples()):
        if r.pair_key in pair_to_label:
            lab = int(pair_to_label[r.pair_key])
            forced_label[pos] = lab
            probs[pos] = float(lab)
            rule_used[pos] = "exact_pair"
            continue

        pos_titles = content_to_pos_titles.get(r.content_key, set())
        if ENABLE_CONTENT_RULE and pos_titles and r.title_key not in pos_titles:
            forced_label[pos] = 0
            probs[pos] = min(probs[pos], CONTENT_RULE_PROB)
            rule_used[pos] = "content_pos_different_title"
            continue

        pos_contents = title_to_pos_contents.get(r.title_key, set())
        if ENABLE_TITLE_RULE and pos_contents and r.content_key not in pos_contents:
            forced_label[pos] = 0
            probs[pos] = min(probs[pos], TITLE_RULE_PROB)
            rule_used[pos] = "title_pos_different_content"

    return probs, rule_used, forced_label

final_test_prob, test_rule, test_forced_label = apply_graph_layer(test, test_meta, train)
final_test_label = (final_test_prob >= meta_t).astype(np.int8)
forced_mask = test_forced_label >= 0
final_test_label[forced_mask] = test_forced_label[forced_mask]

rule_counts = pd.Series(test_rule).value_counts()
print("Test graph-rule coverage:")
display(rule_counts.to_frame("rows"))
print("Forced graph labels:", int(forced_mask.sum()))
print("Predicted label distribution:", dict(pd.Series(final_test_label).value_counts().sort_index()))


Test graph-rule coverage:


,rows
semantic,2460
exact_pair,927
content_pos_different_title,216


Forced graph labels: 1143
Predicted label distribution: {0: np.int64(368), 1: np.int64(3235)}


In [14]:
submission = sample[["id"]].copy()
assert submission.id.tolist() == test.id.tolist(), "sample_submission order does not match test.csv"
submission["label"] = final_test_label.astype(int)

submission_path = OUTPUT_DIR / "submission_v1_1_1.csv"
oof_path = OUTPUT_DIR / "oof_v1_1_1.csv"
test_scores_path = OUTPUT_DIR / "test_scores_v1_1_1.csv"
history_path = OUTPUT_DIR / "training_history_v1_1_1.csv"
config_path = OUTPUT_DIR / "config_v1_1_1.json"

submission.to_csv(submission_path, index=False)
pd.DataFrame({
    "id": train.id,
    "label": train.label.astype(int),
    "semantic_prob": oof_semantic,
    "meta_prob": oof_meta,
}).to_csv(oof_path, index=False)
pd.DataFrame({
    "id": test.id,
    "semantic_prob": test_semantic,
    "meta_prob": test_meta,
    "final_prob": final_test_prob,
    "graph_rule": test_rule,
    "forced_label": test_forced_label,
    "label": final_test_label.astype(int),
}).to_csv(test_scores_path, index=False)

pd.DataFrame(train_history).to_csv(history_path, index=False)
with open(config_path, "w") as f:
    json.dump({
        "version": "v1.1.1",
        "model": MODEL_NAME,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "max_pairwise_per_batch": MAX_PAIRWISE_PER_BATCH,
        "amp_enabled": AMP_ENABLED,
        "gradient_checkpointing": USE_GRAD_CHECKPOINTING,
        "lr": LR,
        "neg_class_weight": NEG_CLASS_WEIGHT,
        "pairwise_lambda": PAIRWISE_LAMBDA,
        "pairwise_margin": PAIRWISE_MARGIN,
        "semantic_oof_macro_f1": float(base_f1),
        "semantic_threshold": float(base_t),
        "meta_oof_macro_f1": float(meta_f1),
        "meta_threshold": float(meta_t),
        "content_rule_precision": float(content_rule_precision),
        "title_rule_precision": float(title_rule_precision),
        "content_rule_enabled": bool(ENABLE_CONTENT_RULE),
        "title_rule_enabled": bool(ENABLE_TITLE_RULE),
        "smoke_test": bool(SMOKE_TEST),
    }, f, indent=2)

print("Saved:")
for p in [submission_path, oof_path, test_scores_path, history_path, config_path]:
    print(" -", p)
display(submission.head())


Saved:
 - /kaggle/working/submission_v1_1_1.csv
 - /kaggle/working/oof_v1_1_1.csv
 - /kaggle/working/test_scores_v1_1_1.csv
 - /kaggle/working/training_history_v1_1_1.csv
 - /kaggle/working/config_v1_1_1.json


,id,label
0,te00000,0
1,te00001,1
2,te00002,1
3,te00003,1
4,te00004,1


## Reading the result

Use **`submission_v1_1.csv`** for Kaggle submission.

For experiment decisions, inspect:

- strict connected-component **Macro F1**,
- **F1 class 0** and **F1 class 1** separately,
- probability quantiles and `mean_p(y0)` vs `mean_p(y1)`,
- confusion matrix,
- epoch runtime / throughput in `training_history_v1_1.csv`,
- graph-rule coverage in `test_scores_v1_1.csv`.

The first V1.1 run should stay at **3 folds × 2 epochs**. Only consider `N_FOLDS=5` after this run is healthy and meaningfully competitive.

Recommended next ablations after V1.1 is stable:

- XLM-R Large NLI as a diversity ensemble member,
- targeted synthetic entity/number/negation hard negatives,
- multi-instance scoring of each evidence chunk separately instead of concatenated evidence,
- multi-seed ensembling,
- finally `N_FOLDS: 3 → 5` for a higher-budget final run.

References motivating the architecture include IndoNLI, FactCC, SummaC, CLIFF, and Falsesum.
